In [1]:
import os
import time
import numpy as np
import pandas as pd
import folium

from initialize_db import get_connection, get_async_connection, get_sch_db_name, get_rt_db_name
from google.protobuf.json_format import MessageToDict

In [2]:
class GTFS:
    def __init__(self, location) -> None:
        self.loctaion = location
        client, _ = get_connection()
        rt_db = client[f"{location}__realtime"]
        sc_db = client[f"{location}__schedule"]

        for i in sc_db.list_collections():
            if i["name"] == "NA":
                continue
            setattr(self, i["name"].split("___")[-1], pd.DataFrame(sc_db[i["name"]].find({})))

        for i in rt_db.list_collections():
            if i["name"] == "NA":
                continue
            setattr(self, i["name"], pd.DataFrame(rt_db[i["name"]].find({})))
        
        client.close()
        

In [3]:
gtfs = GTFS("salt_lake_city")

In [4]:
gtfs.trip_updates.shape

(241438, 754)

In [5]:
vars(gtfs).keys()

dict_keys(['loctaion', 'routes', 'agency', 'calendar_dates', 'stop_times', 'stops', 'calendar', 'shapes', 'trips', 'trip_updates', 'vehicle_positions', 'alerts'])

In [11]:
# lat, lon = gtfs.stops.stop_lat.mean(), gtfs.stops.stop_lon.mean()
# m = folium.Map(location=(lat, lon), tiles="cartodb positron")

# for i, stop in enumerate(gtfs.stops.itertuples()):
#     folium.Marker(
#         location=[stop.stop_lat, stop.stop_lon],
#         tooltip=stop.stop_name,
#         # icon=folium.Icon(icon="circle"),
#     ).add_to(m)
# m

In [81]:
def actual_arrivals_for_trip(trip_id):
    x = gtfs.trip_updates.query(f"trip__trip_id == {trip_id}")
    stops = np.unique(x.loc[:, x.columns.str.contains("__stop_sequence")].values.flatten())
    stops = stops[~np.isnan(stops)]
    x = x.loc[:, ~x.map(lambda x: np.nan if x == "NONE" else x).isnull().all(axis=0)]
    stop_seqs = x.loc[:, x.columns.str.contains("__stop_sequence")]

    aatp = {}
    for i in range(stop_seqs.shape[0]):
        s = stop_seqs[f"stop_time_update_{i}__stop_sequence"]
        pair = s.iloc[::-1].drop_duplicates(keep='first').to_dict()
        pair = {v:k for k,v in pair.items()}
        for k in pair.keys():
            if k not in aatp.keys() and not np.isnan(k):
                aatp[k] = (pair[k], i)
        if len(aatp.keys()) == stops.shape[0]:
            break

    aat = {}
    for k, v in aatp.items():
        ind, col = v
        aat[k] = x.loc[ind, f"stop_time_update_{col}__arrival__time"].item()
    aat = pd.DataFrame([aat]).T.rename(columns={0:"actual_arrivals"}).sort_index()
    return aat

In [86]:
trip_id = np.random.choice(gtfs.trip_updates.trip__trip_id.unique())

In [87]:
x1 = gtfs.stop_times.query(f"trip_id == {trip_id}")[["stop_sequence", "arrival_time"]].rename(columns={"arrival_time": "scheduled_arrival_time"}).set_index("stop_sequence", drop=True)
x2 = actual_arrivals_for_trip(trip_id=trip_id).astype("datetime64[s]").apply(lambda x: x.dt.tz_localize('UTC').dt.tz_convert(gtfs.agency.agency_timezone.item()))
pd.concat([x1, x2.apply(lambda x: x.dt.time)], axis=1)

,scheduled_arrival_time,actual_arrivals
1.0,6:30:00,NaN
2.0,6:31:03,06:30:48
3.0,6:32:12,06:32:05
4.0,6:34:00,06:32:57
5.0,6:35:09,06:35:01
...,...,...
162.0,9:13:29,09:13:45
163.0,9:13:45,09:14:00
164.0,9:14:00,09:14:32
165.0,9:14:32,09:16:00
